# Live Trade Review — XAUUSD Order Block Bot
اطلاعات ترید واقعی از MetaTrader 5 — magic `8088080`

سلول‌ها به ترتیب اجرا شوند. MT5 باید باز و لاگین باشد.

In [61]:
from __future__ import annotations
import os, sys
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:.2f}'.format)

try:
    import MetaTrader5 as mt5
except ImportError:
    sys.exit('pip install MetaTrader5')

MAGIC   = 8088080
SYMBOL  = 'XAUUSD'
REPO    = Path('..').resolve()          # notebooks/ → repo root
LOG_FILE  = REPO / 'logs' / 'XAUUSD.json'
BT_FILE   = REPO / 'notebooks' / 'results' / '08_order_block_reaction' / 'XAUUSD' / 'trades_raw.csv'

## 1 — اتصال به MT5

In [62]:
def mt5_connect():
    kwargs = {}
    for key, env in [('login','MT5_LOGIN'),('password','MT5_PASSWORD'),('server','MT5_SERVER')]:
        v = os.environ.get(env)
        if v:
            kwargs[key] = int(v) if key == 'login' else v
    path = os.environ.get('MT5_TERMINAL_PATH')
    if path:
        kwargs['path'] = path
    if not mt5.initialize(**kwargs):
        raise RuntimeError(f'mt5.initialize failed: {mt5.last_error()}')

mt5_connect()
ti = mt5.terminal_info()
print(f'Connected: {ti.connected}  |  Build: {ti.build}  |  Path: {ti.path}')

Connected: True  |  Build: 5833  |  Path: C:\Program Files\Errante Securities MT5 Terminal


## 2 — اطلاعات اکانت

In [63]:
ai = mt5.account_info()
if ai is None:
    print('account_info() failed:', mt5.last_error())
else:
    print(f'  Login    : {ai.login}')
    print(f'  Server   : {ai.server}')
    print(f'  Name     : {ai.name}')
    print(f'  Currency : {ai.currency}')
    print(f'  Balance  : {ai.balance:,.2f}')
    print(f'  Equity   : {ai.equity:,.2f}')
    print(f'  Margin   : {ai.margin:,.2f}')
    print(f'  Free Mgn : {ai.margin_free:,.2f}')
    print(f'  Profit   : {ai.profit:+.2f}')
    print(f'  Leverage : 1:{ai.leverage}')

  Login    : 13399867
  Server   : ErranteSC-Demo
  Name     : Seyed Reza Moosavi
  Currency : USD
  Balance  : 996.67
  Equity   : 998.18
  Margin   : 4.55
  Free Mgn : 993.63
  Profit   : +1.51
  Leverage : 1:100


## 3 — پوزیشن‌های باز فعلی (magic 8088080)

In [64]:
mt5.symbol_select(SYMBOL, True)
all_pos = mt5.positions_get(symbol=SYMBOL) or []
our_pos = [p for p in all_pos if p.magic == MAGIC]

if not our_pos:
    print('هیچ پوزیشن بازی وجود ندارد.')
else:
    rows = []
    for p in our_pos:
        direction = 'BUY' if p.type == mt5.POSITION_TYPE_BUY else 'SELL'
        open_time = datetime.fromtimestamp(p.time, tz=timezone.utc)
        tick = mt5.symbol_info_tick(SYMBOL)
        cur_price = tick.bid if direction == 'BUY' else tick.ask
        rows.append({
            'ticket'    : p.ticket,
            'direction' : direction,
            'volume'    : p.volume,
            'open_price': p.price_open,
            'cur_price' : cur_price,
            'SL'        : p.sl,
            'TP'        : p.tp,
            'profit'    : p.profit,
            'swap'      : p.swap,
            'open_time' : open_time.strftime('%Y-%m-%d %H:%M'),
            'comment'   : p.comment,
        })
    df_pos = pd.DataFrame(rows)
    display(df_pos)

هیچ پوزیشن بازی وجود ندارد.


## 4 — تاریخچه معاملات بسته‌شده (deals)

In [65]:
# بازه زمانی: از ابتدای ماه جاری تا الان
now   = datetime.now(timezone.utc)
start = now - timedelta(days=30)      # ۳۰ روز گذشته

deals_raw = mt5.history_deals_get(start, now, group=f'*{SYMBOL}*') or []
deals_bot = [d for d in deals_raw if d.magic == MAGIC]

print(f'تعداد کل deals برای {SYMBOL} (30 روز): {len(deals_raw)}')
print(f'تعداد deals ربات (magic={MAGIC}): {len(deals_bot)}')

if not deals_bot:
    print('هیچ deal بسته‌ای یافت نشد.')
else:
    rows = []
    for d in deals_bot:
        entry_map = {0: 'IN', 1: 'OUT', 2: 'INOUT', 3: 'OUT_BY'}
        type_map  = {0: 'BUY', 1: 'SELL', 2: 'BALANCE', 3: 'CREDIT'}
        rows.append({
            'ticket'    : d.ticket,
            'order'     : d.order,
            'position_id': d.position_id,
            'type'      : type_map.get(d.type, d.type),
            'entry'     : entry_map.get(d.entry, d.entry),
            'volume'    : d.volume,
            'price'     : d.price,
            'profit'    : d.profit,
            'swap'      : d.swap,
            'commission': d.commission,
            'time'      : datetime.fromtimestamp(d.time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M'),
            'comment'   : d.comment,
        })
    df_deals = pd.DataFrame(rows).sort_values('time').reset_index(drop=True)
    display(df_deals)

تعداد کل deals برای XAUUSD (30 روز): 12
تعداد deals ربات (magic=8088080): 12


,ticket,order,position_id,type,entry,volume,price,profit,swap,commission,time,comment
0,30016981,38673979,38673979,BUY,IN,0.01,4545.03,0.00,0.00,0.00,2026-05-18 01:25,ob_reaction
1,30020379,38677412,38673979,SELL,OUT,0.01,4530.38,-14.65,0.00,0.00,2026-05-18 03:03,[sl 4530.38]
2,30035976,38692873,38692873,SELL,IN,0.01,4545.46,0.00,0.00,0.00,2026-05-18 07:10,ob_reaction
3,30036728,38693631,38692873,BUY,OUT,0.01,4537.05,8.41,0.00,0.00,2026-05-18 07:24,[tp 4537.58]
4,30039407,38696233,38696233,SELL,IN,0.01,4549.77,0.00,0.00,0.00,2026-05-18 08:30,ob_reaction
5,30040143,38696919,38696233,BUY,OUT,0.01,4538.23,11.54,0.00,0.00,2026-05-18 08:46,[tp 4538.38]
6,30040540,38697303,38697303,BUY,IN,0.01,4535.85,0.00,0.00,0.00,2026-05-18 08:55,ob_reaction
7,30040743,38697505,38697303,SELL,OUT,0.01,4532.42,-3.43,0.00,0.00,2026-05-18 08:58,[sl 4532.42]
8,30042632,38699368,38699368,SELL,IN,0.01,4537.52,0.00,0.00,0.00,2026-05-18 09:30,ob_reaction
9,30044204,38700900,38699368,BUY,OUT,0.01,4547.63,-10.11,0.00,0.00,2026-05-18 09:55,[sl 4547.63]


## 5 — جفت‌سازی ورود/خروج → ترید کامل

In [66]:
def pair_deals(deals_bot: list) -> pd.DataFrame:
    """Match IN and OUT deals by position_id to build complete trade records."""
    entry_map = {0: 'IN', 1: 'OUT', 2: 'INOUT', 3: 'OUT_BY'}
    type_map  = {0: 'BUY', 1: 'SELL'}

    entries = {}   # position_id → entry deal
    trades  = []

    for d in sorted(deals_bot, key=lambda x: x.time):
        etype = entry_map.get(d.entry, '')
        if etype == 'IN':
            entries[d.position_id] = d
        elif etype in ('OUT', 'OUT_BY', 'INOUT') and d.position_id in entries:
            e = entries.pop(d.position_id)
            direction = type_map.get(e.type, str(e.type))
            entry_time = datetime.fromtimestamp(e.time, tz=timezone.utc)
            exit_time  = datetime.fromtimestamp(d.time, tz=timezone.utc)
            profit     = e.profit + d.profit
            swap       = e.swap   + d.swap
            comm       = e.commission + d.commission
            net        = profit + swap + comm
            # SL/TP from the original order
            order_info = mt5.history_orders_get(position=d.position_id) or []
            sl = tp = None
            if order_info:
                for o in order_info:
                    if o.sl > 0: sl = o.sl
                    if o.tp > 0: tp = o.tp
            # Outcome
            if sl and tp:
                if direction == 'BUY':
                    result = 'TP' if d.price >= tp * 0.9999 else ('SL' if d.price <= sl * 1.0001 else 'MANUAL')
                else:
                    result = 'TP' if d.price <= tp * 1.0001 else ('SL' if d.price >= sl * 0.9999 else 'MANUAL')
            else:
                result = 'CLOSED'
            trades.append({
                'position_id': d.position_id,
                'direction'  : direction,
                'volume'     : e.volume,
                'entry_price': e.price,
                'exit_price' : d.price,
                'SL'         : sl,
                'TP'         : tp,
                'result'     : result,
                'profit'     : round(profit, 2),
                'swap'       : round(swap,   2),
                'commission' : round(comm,   2),
                'net_pnl'    : round(net,    2),
                'entry_time' : entry_time.strftime('%Y-%m-%d %H:%M'),
                'exit_time'  : exit_time.strftime('%Y-%m-%d %H:%M'),
                'duration_h' : round((exit_time - entry_time).total_seconds() / 3600, 2),
            })

    # Still-open positions (only entry deal seen)
    for pos_id, e in entries.items():
        direction = type_map.get(e.type, str(e.type))
        entry_time = datetime.fromtimestamp(e.time, tz=timezone.utc)
        order_info = mt5.history_orders_get(position=pos_id) or []
        sl = tp = None
        for o in order_info:
            if o.sl > 0: sl = o.sl
            if o.tp > 0: tp = o.tp
        trades.append({
            'position_id': pos_id,
            'direction'  : direction,
            'volume'     : e.volume,
            'entry_price': e.price,
            'exit_price' : None,
            'SL'         : sl,
            'TP'         : tp,
            'result'     : 'OPEN',
            'profit'     : None,
            'swap'       : None,
            'commission' : None,
            'net_pnl'    : None,
            'entry_time' : entry_time.strftime('%Y-%m-%d %H:%M'),
            'exit_time'  : None,
            'duration_h' : None,
        })

    return pd.DataFrame(trades).sort_values('entry_time').reset_index(drop=True)


df_trades = pair_deals(deals_bot)
print(f'\nتعداد تریدهای کامل (بسته): {len(df_trades[df_trades.result != "OPEN"])}')
print(f'تعداد تریدهای باز:          {len(df_trades[df_trades.result == "OPEN"])}')
print()
display(df_trades)


تعداد تریدهای کامل (بسته): 6
تعداد تریدهای باز:          0



,position_id,direction,volume,entry_price,exit_price,SL,TP,result,profit,swap,commission,net_pnl,entry_time,exit_time,duration_h
0,38673979,BUY,0.01,4545.03,4530.38,4530.38,4565.07,SL,-14.65,0.00,0.00,-14.65,2026-05-18 01:25,2026-05-18 03:03,1.64
1,38692873,SELL,0.01,4545.46,4537.05,4555.70,4537.58,TP,8.41,0.00,0.00,8.41,2026-05-18 07:10,2026-05-18 07:24,0.25
2,38696233,SELL,0.01,4549.77,4538.23,4555.15,4538.38,TP,11.54,0.00,0.00,11.54,2026-05-18 08:30,2026-05-18 08:46,0.28
3,38697303,BUY,0.01,4535.85,4532.42,4532.42,4563.68,SL,-3.43,0.00,0.00,-3.43,2026-05-18 08:55,2026-05-18 08:58,0.06
4,38699368,SELL,0.01,4537.52,4547.63,4547.63,4526.42,SL,-10.11,0.00,0.00,-10.11,2026-05-18 09:30,2026-05-18 09:55,0.43
5,38729143,BUY,0.01,4543.07,4531.91,4531.91,4556.24,SL,-11.16,0.00,0.00,-11.16,2026-05-18 17:50,2026-05-18 18:04,0.23


## 6 — آمار کلی عملکرد

In [67]:
closed = df_trades[df_trades['result'].isin(['TP','SL','MANUAL','CLOSED'])].copy()

if closed.empty:
    print('هنوز هیچ ترید بسته‌ای وجود ندارد.')
else:
    total      = len(closed)
    wins       = len(closed[closed['result'] == 'TP'])
    losses     = len(closed[closed['result'] == 'SL'])
    win_rate   = wins / total * 100 if total else 0
    net_total  = closed['net_pnl'].sum()
    avg_win    = closed.loc[closed['result']=='TP',  'net_pnl'].mean() if wins   else 0
    avg_loss   = closed.loc[closed['result']=='SL',  'net_pnl'].mean() if losses else 0
    profit_factor = (
        closed.loc[closed['net_pnl']>0,'net_pnl'].sum() /
        abs(closed.loc[closed['net_pnl']<0,'net_pnl'].sum())
    ) if closed['net_pnl'].lt(0).any() else float('inf')

    print('━'*50)
    print(f'  تعداد کل تریدها    : {total}')
    print(f'  برد (TP)            : {wins}  ({win_rate:.1f}%)')
    print(f'  باخت (SL)           : {losses}')
    print(f'  میانگین سود هر برد  : {avg_win:+.2f}')
    print(f'  میانگین ضرر هر باخت : {avg_loss:+.2f}')
    print(f'  Profit Factor       : {profit_factor:.2f}')
    print(f'  سود/زیان خالص کل   : {net_total:+.2f}')
    print('━'*50)
    print()
    print('نتایج به تفکیک جهت:')
    print(closed.groupby(['direction','result'])['net_pnl'].agg(['count','sum','mean']).round(2))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  تعداد کل تریدها    : 6
  برد (TP)            : 2  (33.3%)
  باخت (SL)           : 4
  میانگین سود هر برد  : +9.97
  میانگین ضرر هر باخت : -9.84
  Profit Factor       : 0.51
  سود/زیان خالص کل   : -19.40
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

نتایج به تفکیک جهت:
                  count    sum   mean
direction result                     
BUY       SL          3 -29.24  -9.75
SELL      SL          1 -10.11 -10.11
          TP          2  19.95   9.98


## 7 — مقایسه لایو با بک‌تست (بر اساس ob_time)

In [68]:
import json

# ---- استخراج سیگنال‌های ربات از لاگ JSON-Lines ----
log_events = []
if LOG_FILE.exists():
    with LOG_FILE.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                log_events.append(json.loads(line))

placed  = [e for e in log_events if e['event'] == 'order_placed']
signals = {e['ts'][:16]: e for e in log_events if e['event'] == 'signal'}  # key by minute

# لیست تریدهای لایو با اطلاعات سیگنال
live_rows = []
for p in placed:
    # پیدا کردن سیگنال متناظر (همان دقیقه)
    sig = signals.get(p['ts'][:16], {})
    live_rows.append({
        'ticket'     : p.get('ticket'),
        'direction'  : p.get('direction'),
        'entry_live' : p.get('ob_high') if p.get('direction')=='BUY' else p.get('ob_low'),
        'SL_live'    : p.get('sl'),
        'TP_live'    : p.get('tp'),
        'ob_time'    : sig.get('ob_time', p.get('ob_time','')),
        'retest_live': sig.get('retest_time',''),
        'placed_at'  : p['ts'][:16],
    })

df_live = pd.DataFrame(live_rows)
print(f'تریدهای ثبت‌شده در لاگ: {len(df_live)}')
display(df_live)

# ---- بک‌تست ----
if BT_FILE.exists():
    df_bt = pd.read_csv(BT_FILE)
    df_bt['ob_time'] = pd.to_datetime(df_bt['ob_time'], utc=True).astype(str).str[:19]
    df_bt['entry_time'] = pd.to_datetime(df_bt['entry_time'], utc=True)
    df_bt_recent = df_bt[df_bt['entry_time'] >= pd.Timestamp('2026-05-17', tz='UTC')].copy()

    print(f'\nتریدهای بک‌تست از 2026-05-17 به بعد:')
    display(df_bt_recent[['direction','entry_price','sl','tp','result','entry_time','retest_time','ob_time']])

    # ---- Merge ----
    df_live['ob_time_key'] = df_live['ob_time'].str[:19]
    merged = df_live.merge(
        df_bt_recent[['ob_time','entry_price','sl','tp','result']].rename(columns={
            'entry_price':'entry_bt','sl':'SL_bt','tp':'TP_bt','result':'result_bt'
        }),
        left_on='ob_time_key', right_on='ob_time', how='left'
    ).drop(columns=['ob_time_y'], errors='ignore')

    merged['entry_match'] = (merged['entry_live'] - merged['entry_bt']).abs() < 0.02
    merged['SL_match']    = (merged['SL_live']    - merged['SL_bt']).abs() < 0.02
    merged['TP_match']    = (merged['TP_live']     - merged['TP_bt']).abs() < 0.02

    print('\nمقایسه لایو vs بک‌تست:')
    display(merged[[
        'ticket','direction',
        'entry_live','entry_bt','entry_match',
        'SL_live','SL_bt','SL_match',
        'TP_live','TP_bt','TP_match',
        'result_bt','ob_time_key'
    ]])
else:
    print(f'فایل بک‌تست پیدا نشد: {BT_FILE}')

تریدهای ثبت‌شده در لاگ: 8


,ticket,direction,entry_live,SL_live,TP_live,ob_time,retest_live,placed_at
0,38673979,BUY,4541.94,4530.38,4565.06,2026-05-15 16:55:00+00:00,2026-05-18 01:20:00+00:00,2026-05-17T22:25
1,38692873,SELL,4549.66,4555.70,4537.58,2026-05-15 22:35:00+00:00,2026-05-18 07:05:00+00:00,2026-05-18T04:10
2,38696233,SELL,4549.56,4555.15,4538.38,2026-05-18 01:45:00+00:00,2026-05-18 08:25:00+00:00,2026-05-18T05:30
3,38697303,BUY,4542.84,4532.42,4563.68,2026-05-18 08:00:00+00:00,2026-05-18 08:50:00+00:00,2026-05-18T05:55
4,38699368,SELL,4540.56,4547.63,4526.42,2026-05-18 08:35:00+00:00,2026-05-18 09:25:00+00:00,2026-05-18T06:30
5,38729143,BUY,4540.02,4531.91,4556.24,2026-05-18 14:20:00+00:00,2026-05-18 17:45:00+00:00,2026-05-18T14:50
6,38757741,SELL,4546.22,4549.92,4538.82,2026-05-19 06:50:00+00:00,2026-05-19 08:10:00+00:00,2026-05-19T05:15
7,38763571,SELL,4551.06,4560.05,4519.76,2026-05-19 05:40:00+00:00,2026-05-19 09:25:00+00:00,2026-05-19T06:30



تریدهای بک‌تست از 2026-05-17 به بعد:


,direction,entry_price,sl,tp,result,entry_time,retest_time,ob_time
284,SELL,4549.66,4555.70,4537.58,TP,2026-05-18 01:35:00+00:00,2026-05-18 01:30:00+00:00,2026-05-15 22:35:00
285,BUY,4541.37,4537.38,4549.35,TP,2026-05-18 01:25:00+00:00,2026-05-18 01:20:00+00:00,2026-05-15 23:45:00
286,SELL,4549.56,4555.15,4538.38,TP,2026-05-18 02:00:00+00:00,2026-05-18 01:55:00+00:00,2026-05-18 01:45:00
287,SELL,4523.86,4530.38,4510.82,TP,2026-05-18 03:25:00+00:00,2026-05-18 03:20:00+00:00,2026-05-18 03:10:00
288,BUY,4542.84,4532.42,4563.68,SL,2026-05-18 08:25:00+00:00,2026-05-18 08:20:00+00:00,2026-05-18 08:00:00
289,SELL,4540.56,4547.63,4526.42,SL,2026-05-18 08:50:00+00:00,2026-05-18 08:45:00+00:00,2026-05-18 08:35:00
290,BUY,4540.02,4531.41,4557.24,TP,2026-05-18 17:45:00+00:00,2026-05-18 17:40:00+00:00,2026-05-18 14:20:00
291,BUY,4565.57,4562.59,4571.53,SL,2026-05-19 04:00:00+00:00,2026-05-19 03:55:00+00:00,2026-05-18 23:50:00
292,SELL,4584.12,4586.96,4578.44,TP,2026-05-19 03:05:00+00:00,2026-05-19 03:00:00+00:00,2026-05-19 02:40:00
293,BUY,4559.80,4553.58,4572.24,SL,2026-05-19 05:05:00+00:00,2026-05-19 05:00:00+00:00,2026-05-19 04:20:00



مقایسه لایو vs بک‌تست:


,ticket,direction,entry_live,entry_bt,entry_match,SL_live,SL_bt,SL_match,TP_live,TP_bt,TP_match,result_bt,ob_time_key
0,38673979,BUY,4541.94,NaN,False,4530.38,NaN,False,4565.06,NaN,False,NaN,2026-05-15 16:55:00
1,38692873,SELL,4549.66,4549.66,True,4555.70,4555.70,True,4537.58,4537.58,True,TP,2026-05-15 22:35:00
2,38696233,SELL,4549.56,4549.56,True,4555.15,4555.15,True,4538.38,4538.38,True,TP,2026-05-18 01:45:00
3,38697303,BUY,4542.84,4542.84,True,4532.42,4532.42,True,4563.68,4563.68,True,SL,2026-05-18 08:00:00
4,38699368,SELL,4540.56,4540.56,True,4547.63,4547.63,True,4526.42,4526.42,True,SL,2026-05-18 08:35:00
5,38729143,BUY,4540.02,4540.02,True,4531.91,4531.41,False,4556.24,4557.24,False,TP,2026-05-18 14:20:00
6,38757741,SELL,4546.22,4546.22,True,4549.92,4549.92,True,4538.82,4538.82,True,TP,2026-05-19 06:50:00
7,38763571,SELL,4551.06,4551.06,True,4560.05,4560.05,True,4519.76,4533.08,False,TP,2026-05-19 05:40:00


## 10 — وضعیت همه سفارش‌های ثبت‌شده در لاگ (Filled vs Cancelled)

In [69]:
STATE_MAP = {
    1: 'PLACED',
    2: 'CANCELED',
    3: 'PARTIAL',
    4: 'FILLED',
    5: 'REJECTED',
    6: 'EXPIRED',
}
TYPE_MAP = {
    0: 'BUY_MARKET',
    1: 'SELL_MARKET',
    2: 'BUY_LIMIT',
    3: 'SELL_LIMIT',
    4: 'BUY_STOP',
    5: 'SELL_STOP',
}

log_tickets = [int(p['ticket']) for p in placed if p.get('ticket')]
print(f'تعداد سفارش‌های ثبت‌شده در لاگ: {len(log_tickets)}')
print()

rows = []
for ticket in log_tickets:
    orders = mt5.history_orders_get(ticket=ticket) or []
    if not orders:
        rows.append({'ticket': ticket, 'state': 'NOT FOUND', 'order_type': '-',
                     'price_open': '-', 'volume_initial': '-', 'volume_current': '-',
                     'sl': '-', 'tp': '-', 'time_setup': '-', 'time_done': '-', 'comment': '-'})
        continue
    for o in orders:
        rows.append({
            'ticket'        : ticket,
            'state'         : STATE_MAP.get(o.state, o.state),
            'order_type'    : TYPE_MAP.get(o.type, o.type),
            'price_open'    : o.price_open,
            'volume_initial': o.volume_initial,
            'volume_current': o.volume_current,
            'sl'            : o.sl,
            'tp'            : o.tp,
            'time_setup'    : datetime.fromtimestamp(o.time_setup, tz=timezone.utc).strftime('%Y-%m-%d %H:%M'),
            'time_done'     : datetime.fromtimestamp(o.time_done, tz=timezone.utc).strftime('%Y-%m-%d %H:%M') if o.time_done else '-',
            'comment'       : o.comment,
        })

df_order_states = pd.DataFrame(rows)
display(df_order_states)

print()
print('خلاصه وضعیت:')
counts = df_order_states['state'].value_counts()
print(counts.to_string())
print()
filled_count   = counts.get('FILLED',   0)
canceled_count = counts.get('CANCELED', 0)
other_count    = len(df_order_states) - filled_count - canceled_count
print(f'  FILLED   (اجراشده)  : {filled_count}   ← تبدیل به deal شده‌اند')
print(f'  CANCELED (لغوشده)   : {canceled_count}   ← فقط فلش روی چارت، بدون deal')
print(f'  سایر                : {other_count}')


تعداد سفارش‌های ثبت‌شده در لاگ: 8



,ticket,state,order_type,price_open,volume_initial,volume_current,sl,tp,time_setup,time_done,comment
0,38673979,FILLED,BUY_MARKET,0.00,0.01,0.00,4530.38,4565.07,2026-05-18 01:25,2026-05-18 01:25,ob_reaction
1,38692873,FILLED,SELL_MARKET,0.00,0.01,0.00,4555.70,4537.58,2026-05-18 07:10,2026-05-18 07:10,ob_reaction
2,38696233,FILLED,SELL_MARKET,0.00,0.01,0.00,4555.15,4538.38,2026-05-18 08:30,2026-05-18 08:30,ob_reaction
3,38697303,FILLED,BUY_MARKET,0.00,0.01,0.00,4532.42,4563.68,2026-05-18 08:55,2026-05-18 08:55,ob_reaction
4,38699368,FILLED,SELL_MARKET,0.00,0.01,0.00,4547.63,4526.42,2026-05-18 09:30,2026-05-18 09:30,ob_reaction
5,38729143,FILLED,BUY_MARKET,0.00,0.01,0.00,4531.91,4556.24,2026-05-18 17:50,2026-05-18 17:50,ob_reaction
6,38757741,FILLED,SELL_MARKET,0.00,0.02,0.00,4549.92,4538.81,2026-05-19 08:15,2026-05-19 08:15,ob_reaction
7,38763571,FILLED,SELL_MARKET,0.00,0.01,0.00,4560.05,4519.76,2026-05-19 09:30,2026-05-19 09:30,ob_reaction



خلاصه وضعیت:
state
FILLED    8

  FILLED   (اجراشده)  : 8   ← تبدیل به deal شده‌اند
  CANCELED (لغوشده)   : 0   ← فقط فلش روی چارت، بدون deal
  سایر                : 0


## 11 — تحلیل کامل: سود/زیان هر ترید + مقایسه با بک‌تست

In [70]:
import json
from pathlib import Path

# ---- بارگذاری لاگ (اگر از قبل بارگذاری نشده) ----
if 'placed' not in dir() or not placed:
    log_events = []
    if LOG_FILE.exists():
        with LOG_FILE.open(encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    log_events.append(json.loads(line))
    placed = [e for e in log_events if e['event'] == 'order_placed']

# ---- دریافت deals برای هر position مستقیماً ----
now_q = datetime.now(timezone.utc)
start_q = now_q - timedelta(days=60)

live_trades = []
for p in placed:
    pos_id = int(p['ticket'])
    pos_deals = mt5.history_deals_get(position=pos_id) or []

    entry_deal = next((d for d in pos_deals if d.entry == 0), None)  # IN
    exit_deal  = next((d for d in pos_deals if d.entry == 1), None)  # OUT

    direction = p.get('direction', '')
    entry_log_price = p.get('ob_high') if direction == 'BUY' else p.get('ob_low')

    if entry_deal and exit_deal:
        net_pnl = round(entry_deal.profit + exit_deal.profit + entry_deal.swap + exit_deal.swap
                        + entry_deal.commission + exit_deal.commission, 2)
        result_live = 'TP' if (
            (direction == 'BUY'  and exit_deal.price >= p.get('tp', 0) * 0.9999) or
            (direction == 'SELL' and exit_deal.price <= p.get('tp', 0) * 1.0001)
        ) else 'SL' if (
            (direction == 'BUY'  and exit_deal.price <= p.get('sl', 0) * 1.0001) or
            (direction == 'SELL' and exit_deal.price >= p.get('sl', 0) * 0.9999)
        ) else 'MANUAL'
        live_trades.append({
            'ticket'      : pos_id,
            'direction'   : direction,
            'entry_price' : entry_deal.price,
            'exit_price'  : exit_deal.price,
            'SL'          : p.get('sl'),
            'TP'          : p.get('tp'),
            'result_live' : result_live,
            'net_pnl'     : net_pnl,
            'entry_time'  : datetime.fromtimestamp(entry_deal.time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M'),
            'exit_time'   : datetime.fromtimestamp(exit_deal.time,  tz=timezone.utc).strftime('%Y-%m-%d %H:%M'),
            'ob_time'     : str(p.get('ob_time', ''))[:19],
        })
    elif entry_deal:
        # هنوز باز است
        tick = mt5.symbol_info_tick(SYMBOL + '.i') or mt5.symbol_info_tick(SYMBOL)
        cur = tick.bid if direction == 'BUY' else tick.ask
        live_trades.append({
            'ticket'      : pos_id,
            'direction'   : direction,
            'entry_price' : entry_deal.price,
            'exit_price'  : None,
            'SL'          : p.get('sl'),
            'TP'          : p.get('tp'),
            'result_live' : 'OPEN',
            'net_pnl'     : round((cur - entry_deal.price) * entry_deal.volume * (1 if direction=='BUY' else -1) * 100, 2),
            'entry_time'  : datetime.fromtimestamp(entry_deal.time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M'),
            'exit_time'   : None,
            'ob_time'     : str(p.get('ob_time', ''))[:19],
        })
    else:
        live_trades.append({
            'ticket': pos_id, 'direction': direction,
            'entry_price': None, 'exit_price': None,
            'SL': p.get('sl'), 'TP': p.get('tp'),
            'result_live': 'NO DEALS', 'net_pnl': None,
            'entry_time': None, 'exit_time': None,
            'ob_time': str(p.get('ob_time', ''))[:19],
        })

df_live_full = pd.DataFrame(live_trades)

# ---- بک‌تست ----
bt_map = {}
if BT_FILE.exists():
    df_bt_all = pd.read_csv(BT_FILE)
    df_bt_all['ob_time_key'] = pd.to_datetime(df_bt_all['ob_time'], utc=True).astype(str).str[:19]
    for _, row in df_bt_all.iterrows():
        bt_map[row['ob_time_key']] = row

# ---- Merge ----
records = []
for _, row in df_live_full.iterrows():
    bt = bt_map.get(row['ob_time'], {})
    records.append({
        'ticket'       : row['ticket'],
        'direction'    : row['direction'],
        'entry_live'   : row['entry_price'],
        'entry_bt'     : bt.get('entry_price'),
        'exit_live'    : row['exit_price'],
        'SL'           : row['SL'],
        'TP'           : row['TP'],
        'result_live'  : row['result_live'],
        'result_bt'    : bt.get('result', 'NO MATCH'),
        'pnl_live'     : row['net_pnl'],
        'pnl_bt'       : bt.get('pnl'),
        'entry_time'   : row['entry_time'],
        'exit_time'    : row['exit_time'],
        'ob_time'      : row['ob_time'],
        'match'        : row['ob_time'] in bt_map,
    })

df_cmp = pd.DataFrame(records)

print('=' * 70)
print('  سود/زیان هر ترید لایو + مقایسه با بک‌تست')
print('=' * 70)
display(df_cmp[[
    'ticket','direction','entry_live','exit_live',
    'result_live','result_bt','pnl_live','pnl_bt','match','entry_time','exit_time'
]])

print()
closed_live = df_cmp[df_cmp['result_live'].isin(['TP','SL','MANUAL'])]
if not closed_live.empty:
    total     = len(closed_live)
    wins_live = (closed_live['result_live'] == 'TP').sum()
    net_live  = closed_live['pnl_live'].sum()
    print(f'  تریدهای بسته‌شده  : {total}')
    print(f'  برد (TP)           : {wins_live}  ({wins_live/total*100:.0f}%)')
    print(f'  باخت (SL)          : {(closed_live["result_live"]=="SL").sum()}')
    print(f'  سود/زیان خالص کل  : {net_live:+.2f} $')
    print()
    matched_bt = df_cmp[df_cmp['match']]
    if not matched_bt.empty:
        print('  مقایسه نتیجه لایو vs بک‌تست (تریدهای match‌شده):')
        for _, r in matched_bt.iterrows():
            icon = '✅' if r['result_live'] == r['result_bt'] else '❌'
            print(f'    {icon}  ticket {r["ticket"]} | live={r["result_live"]} | bt={r["result_bt"]} | pnl={r["pnl_live"]:+.2f}$')


  سود/زیان هر ترید لایو + مقایسه با بک‌تست


,ticket,direction,entry_live,exit_live,result_live,result_bt,pnl_live,pnl_bt,match,entry_time,exit_time
0,38673979,BUY,4545.03,4530.38,SL,SL,-14.65,None,True,2026-05-18 01:25,2026-05-18 03:03
1,38692873,SELL,4545.46,4537.05,TP,TP,8.41,None,True,2026-05-18 07:10,2026-05-18 07:24
2,38696233,SELL,4549.77,4538.23,TP,TP,11.54,None,True,2026-05-18 08:30,2026-05-18 08:46
3,38697303,BUY,4535.85,4532.42,SL,SL,-3.43,None,True,2026-05-18 08:55,2026-05-18 08:58
4,38699368,SELL,4537.52,4547.63,SL,SL,-10.11,None,True,2026-05-18 09:30,2026-05-18 09:55
5,38729143,BUY,4543.07,4531.91,SL,TP,-11.16,None,True,2026-05-18 17:50,2026-05-18 18:04
6,38757741,SELL,4544.80,4538.57,TP,TP,12.46,None,True,2026-05-19 08:15,2026-05-19 08:22
7,38763571,SELL,4546.98,NaN,OPEN,TP,2.21,None,True,2026-05-19 09:30,NaN



  تریدهای بسته‌شده  : 7
  برد (TP)           : 3  (43%)
  باخت (SL)          : 4
  سود/زیان خالص کل  : -6.94 $

  مقایسه نتیجه لایو vs بک‌تست (تریدهای match‌شده):
    ✅  ticket 38673979 | live=SL | bt=SL | pnl=-14.65$
    ✅  ticket 38692873 | live=TP | bt=TP | pnl=+8.41$
    ✅  ticket 38696233 | live=TP | bt=TP | pnl=+11.54$
    ✅  ticket 38697303 | live=SL | bt=SL | pnl=-3.43$
    ✅  ticket 38699368 | live=SL | bt=SL | pnl=-10.11$
    ❌  ticket 38729143 | live=SL | bt=TP | pnl=-11.16$
    ✅  ticket 38757741 | live=TP | bt=TP | pnl=+12.46$
    ❌  ticket 38763571 | live=OPEN | bt=TP | pnl=+2.21$


## 8 — وضعیت آخرین کندل‌های M5 (دیتای زنده)

In [71]:
symbol_mt5 = SYMBOL
# امتحان با پسوند .i در صورت نیاز
si = mt5.symbol_info(symbol_mt5)
if si is None:
    symbol_mt5 = SYMBOL + '.i'
    mt5.symbol_select(symbol_mt5, True)
    si = mt5.symbol_info(symbol_mt5)

if si is None:
    print('سیمبول پیدا نشد')
else:
    tick = mt5.symbol_info_tick(symbol_mt5)
    print(f'Symbol : {symbol_mt5}')
    print(f'Bid    : {tick.bid:.2f}  |  Ask: {tick.ask:.2f}  |  Spread: {tick.ask-tick.bid:.2f}')
    print(f'Time   : {datetime.fromtimestamp(tick.time, tz=timezone.utc)}')
    print()

    r = mt5.copy_rates_from_pos(symbol_mt5, mt5.TIMEFRAME_M5, 0, 10)
    if r is not None:
        df_m5 = pd.DataFrame(r)
        df_m5['time'] = pd.to_datetime(df_m5['time'], unit='s', utc=True)
        print('آخرین ۱۰ کندل M5:')
        display(df_m5[['time','open','high','low','close','tick_volume']])

Symbol : XAUUSD.i
Bid    : 4544.56  |  Ask: 4544.77  |  Spread: 0.21
Time   : 2026-05-19 10:38:02+00:00

آخرین ۱۰ کندل M5:


,time,open,high,low,close,tick_volume
0,2026-05-19 09:50:00+00:00,4543.14,4547.38,4543.14,4546.74,924
1,2026-05-19 09:55:00+00:00,4546.73,4549.60,4546.73,4548.14,1013
2,2026-05-19 10:00:00+00:00,4547.19,4548.73,4544.08,4547.85,1145
3,2026-05-19 10:05:00+00:00,4547.86,4555.24,4546.82,4552.98,1099
4,2026-05-19 10:10:00+00:00,4552.99,4553.86,4547.15,4548.65,1056
5,2026-05-19 10:15:00+00:00,4548.64,4550.04,4546.70,4548.66,985
6,2026-05-19 10:20:00+00:00,4548.79,4549.96,4544.95,4547.01,981
7,2026-05-19 10:25:00+00:00,4547.03,4548.28,4543.08,4548.28,971
8,2026-05-19 10:30:00+00:00,4548.27,4549.10,4543.79,4544.06,980
9,2026-05-19 10:35:00+00:00,4544.03,4547.70,4543.92,4544.56,617


## 9 — قطع اتصال

In [72]:
mt5.shutdown()
print('MT5 disconnected.')

MT5 disconnected.
